# ISS decoding with ISTDECO

This notebook runs ISTDECO as a joint spot detector and barcode decoder. It uses the same registration, white-top-hat filtering, and channel normalization as the standard Starfish workflow, but it bypasses both BlobDetector/Spotiflow and the Starfish/PoSTcode decoders.

SpaceTx is the common input format. If it already exists, leave `CREATE_SPACETX = False`. Start with one region and inspect the threshold/QC plots before launching the full experiment.

## Environment

Install the CUDA build of PyTorch appropriate for the Ubuntu machine first. From the repository root, install the project and all decoder integrations with:

```bash
python -m pip install -U "./ISS_decoding[postcode,spotiflow,istdeco]"
```

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import torch
import istdeco

from ISS_decoding import SpaceTx_format as STX
from ISS_decoding import decoding as DEC

print("PyTorch:", torch.__version__)
print("PyTorch CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("ISTDECO:", istdeco.__version__)

## Paths and experiment layout

`EXPERIMENT_ROOT` must contain `R1`, `R2`, ... directories. With no separate output root, SpaceTx and decoded outputs are written below each region's `decoding/` directory.

In [ ]:
EXPERIMENT_ROOT = Path("/mnt/DATA/path/to/experiment")
CODEBOOK_CSV = Path("/mnt/DATA/path/to/codebook.csv")
OUTPUT_ROOT = None  # or Path("/mnt/DATA/path/to/decoding_outputs")
REGIONS_TO_PROCESS = [1]

CREATE_SPACETX = False
RUN_DECODING = False

PIXEL_TO_UM = 1.0
CHANNELS = ["DAPI", "Cy3", "Cy5", "AF750", "AF488"]
DECODING_CHANNELS = ["AF750", "Cy5", "Cy3", "AF488"]
NUCLEI_CHANNEL = "DAPI"
USE_CARE_IMAGES = False

## Optional: create SpaceTx

Run this only when `experiment.json` and `codebook.json` have not already been generated. ISTDECO does not need a second image or codebook format: the adapter reads both directly from SpaceTx.

In [ ]:
if CREATE_SPACETX:
    STX.make_spacetx_format(
        input_dir=EXPERIMENT_ROOT,
        codebook_csv=CODEBOOK_CSV,
        regions_to_process=REGIONS_TO_PROCESS,
        output_dir_prefix=OUTPUT_ROOT,
        pixel_to_um=PIXEL_TO_UM,
        channels=CHANNELS,
        DO_decorators=DECODING_CHANNELS,
        nuclei_channel=NUCLEI_CHANNEL,
        CARE=USE_CARE_IMAGES,
    )
else:
    print("SpaceTx creation is disabled; existing SpaceTx files will be used.")

## ISTDECO settings

The defaults follow the original ISS example (`99th` image percentile and quality above `0.5`). `tile_size` bounds GPU memory. `overlap=None` chooses a PSF-safe halo automatically and removes duplicate seam detections by retaining only non-overlapping tile cores.

`istdeco_quality` is a filtering score, not a calibrated probability. For initial tuning, compare several intensity percentiles and quality thresholds on one representative region.

In [ ]:
ISTDECO_KWARGS = {
    "sigma": 1.2,
    "background": 1e-8,
    "scale": 1.0,
    "niter": 75,
    "acceleration": 1.0,
    "suppress_radius": 1,
    "tile_size": (512, 512),
    "overlap": None,
    "intensity_percentile": 99.0,
    "intensity_threshold": None,
    "quality_threshold": 0.5,
    "device": "auto",
    "z_projection": "max",
}

PIPELINE_KWARGS = {
    "register": False,
    "register_dapi": False,
    "masking_radius": 15,
    "normalization_method": "MH",
}

In [ ]:
if RUN_DECODING:
    DEC.process_experiment(
        input_dir=EXPERIMENT_ROOT,
        regions_to_process=REGIONS_TO_PROCESS,
        output_dir_prefix=OUTPUT_ROOT,
        decode_mode="ISTDECO",
        istdeco_kwargs=ISTDECO_KWARGS,
        **PIPELINE_KWARGS,
    )
else:
    print("Decoding is disabled. Set RUN_DECODING = True when ready.")

## Inspect the result

Parquet is canonical; CSV is written alongside it for compatibility. Adjust `RESULT_REGION` if you processed another region.

In [ ]:
RESULT_REGION = "R1"
base = OUTPUT_ROOT if OUTPUT_ROOT is not None else EXPERIMENT_ROOT
result_dir = base / RESULT_REGION / "decoding" / "2_decoded_istdeco"
result_path = result_dir / f"{RESULT_REGION}_decoded_istdeco.parquet"

if result_path.exists():
    decoded = pd.read_parquet(result_path)
    display(decoded.head())
    print(f"{len(decoded):,} accepted transcripts")
    print(decoded["target"].value_counts().head(20))
else:
    print("No result yet:", result_path)

In [ ]:
if result_path.exists() and len(decoded):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].hist(decoded["istdeco_intensity"], bins=80)
    axes[0].set(title="Accepted intensity", xlabel="ISTDECO intensity")
    axes[1].hist(decoded["istdeco_quality"], bins=80)
    axes[1].set(title="Accepted quality", xlabel="ISTDECO quality")
    axes[2].scatter(decoded["x"], decoded["y"], s=1, alpha=0.5)
    axes[2].invert_yaxis()
    axes[2].set(title="Decoded spot positions", xlabel="x (pixels)", ylabel="y (pixels)")
    plt.tight_layout()

## Reproducibility

Each productive run writes XML and JSON manifests containing the effective ISTDECO settings, installed version, pinned fork commit, coordinate units, and output paths. Per-FOV Parquet files in `tiles/` are restart checkpoints; delete only the specific checkpoint(s) you intentionally want to recompute. Dense barcode maps are intentionally not retained because their storage cost is very large.

In [ ]:
manifests = sorted(result_dir.glob("decoding_run_*.json")) if result_dir.exists() else []
if manifests:
    manifest = json.loads(manifests[-1].read_text())
    print(json.dumps(manifest, indent=2))
else:
    print("No ISTDECO run manifest found.")